# Experiment: Holy QOW Post-6A Steps 0–2 Evidence

**Objective:** reproduce the frozen-artifact checks, conditional analysis, held-out evaluation, D=6 repeatability analysis, and figures from saved evidence.

**Safety boundary:** this notebook does not submit Classiq jobs, overwrite quantum runs, implement Step 3, or change the frozen model. The four new quantum runs are already saved.

**Success criteria:** all gates pass; matched-valid batches use the actual QAOA one-hot count; held-out scenarios remain frozen; all five optimizer seeds, including poor seeds, appear in the final tables.

In [ ]:
# Setup: locate the repository and use its verified Python environment.
from __future__ import annotations

import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

def find_implementation_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'post6a').is_dir() and (candidate / 'stage6a').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook from the Holy QOW implementation checkout')

ROOT = find_implementation_root(Path.cwd())
verified_python = ROOT.parent / '.venv-classiq' / 'Scripts' / 'python.exe'
PYTHON = str(verified_python if verified_python.is_file() else Path(sys.executable))
ARTIFACTS = ROOT / 'artifacts' / 'post6a'

def run_script(relative_path: str) -> str:
    completed = subprocess.run(
        [PYTHON, str(ROOT / relative_path)],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
        env={**os.environ, 'MPLBACKEND': 'Agg', 'MPLCONFIGDIR': str(ARTIFACTS / '.mplconfig')},
    )
    lines = [line for line in completed.stdout.splitlines() if line.strip()]
    return lines[-1] if lines else 'completed'

{'root': str(ROOT), 'python': PYTHON, 'artifact_root': str(ARTIFACTS)}

## Plan

1. Reprocess frozen D=4–6 evidence and verify recorded summaries.
2. Recompute Step 0 conditional and matched-valid metrics without quantum execution.
3. Evaluate the pre-frozen held-out manifest and fixed routes.
4. Reprocess all five saved D=6 seeds and deterministic controls.
5. Regenerate figures and optionally rerun the three test suites.

## Gate 0A — Frozen artifact consistency

In [ ]:
run_script('post6a/scripts/verify_frozen_baseline.py')
baseline = json.loads((ARTIFACTS / 'baseline_verification.json').read_text())
{
    'status': baseline['status'],
    'decoded_shots': {row['D']: row['decoded_shots'] for row in baseline['runs']},
    'discrepancies': baseline['discrepancies'],
}

## Step 0 — Conditional distribution

Each random-valid null batch contains exactly the number of one-hot QAOA shots observed at that D.

In [ ]:
run_script('post6a/scripts/run_step0_conditional.py')
conditional = pd.read_csv(ARTIFACTS / 'conditional' / 'conditional_metrics.csv')
conditional[[
    'D', 'one_hot_count', 'joint_feasible_count', 'near_optimal_count',
    'p_one_hot', 'p_joint_feasible', 'p_feasible_given_onehot',
    'p_near_optimal_given_onehot', 'best_feasible_objective_gap',
]]

In [ ]:
matched0 = pd.read_csv(ARTIFACTS / 'conditional' / 'matched_valid_summary.csv')
matched0[[
    'D', 'matched_batch_size',
    'empirical_p_random_best_gap_at_least_as_good',
    'empirical_p_random_best_regret_at_least_as_good',
    'empirical_p_random_feasible_fraction_at_least_as_high',
    'empirical_p_random_near_fraction_at_least_as_high',
]]

**Step 0 result:** one-hot access is the dominant multiplicative loss. Conditional superiority is not established: every matched-valid empirical p-value is above 0.19.

## Step 1 — Held-out resilience

The scenario and route manifests were frozen before comparison. This cell evaluates those frozen inputs; it does not regenerate or tune them.

In [ ]:
run_script('post6a/scripts/run_step1_heldout.py')
heldout = pd.read_csv(ARTIFACTS / 'heldout' / 'heldout_route_summary.csv')
heldout[[
    'route_id', 'assignment', 'mean_regret', 'worst_case_regret',
    'survival_rate', 'violating_scenarios', 'maximum_overflow',
    'mean_weighted_latency',
]]

**Step 1 result:** the static uniform route is the only 24/24 survivor. Exact-adaptive, QAOA-adaptive, and minimax are the same assignment and underperform the static route on survival and regret.

## Step 2 — D=6 repeatability from saved raw runs

This invokes post-processing only. It does not execute Classiq.

In [ ]:
run_script('post6a/scripts/process_step2_repeatability.py')
repeat = pd.read_csv(ARTIFACTS / 'repeatability' / 'd6_repeatability.csv')
qaoa = repeat[repeat['method'].eq('qaoa')]
qaoa[[
    'optimizer_seed', 'one_hot_count', 'joint_feasible_count',
    'near_optimal_count', 'p_one_hot', 'p_joint_feasible',
    'p_feasible_given_onehot', 'p_near_optimal_given_onehot',
    'best_feasible_objective_gap', 'best_feasible_worst_case_regret',
    'execution_runtime_seconds', 'execution_status',
]]

In [ ]:
comparison = json.loads((ARTIFACTS / 'repeatability' / 'd6_control_comparison_summary.json').read_text())
comparison

**Step 2 result:** QAOA beats its random-bit control on best feasible gap and regret in 5/5 pairs, but one-hot counts range from 5 to 63 and no matched-valid conditional comparison is significant.

## Figures

In [ ]:
run_script('post6a/scripts/generate_post6a_plots.py')
for figure in sorted((ARTIFACTS / 'figures').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure), width=900))

## Optional verification tests

The Stage 6A suite includes slower exact-enumeration checks and may take several minutes.

In [ ]:
def run_tests(path: str) -> str:
    completed = subprocess.run(
        [PYTHON, '-m', 'pytest', path, '-q', '--disable-warnings'],
        cwd=ROOT, check=True, capture_output=True, text=True,
    )
    return completed.stdout.strip().splitlines()[-1]

{
    'v1.1': run_tests('tests'),
    'stage6a': run_tests('stage6a/tests'),
    'post6a': run_tests('post6a/tests'),
}

## Decision

The previous Stage 6A interpretation is **weakened, not overturned**. The random-bit comparison is stronger, but conditional concentration and adaptive held-out resilience are not supported.

**STEP 3 RECOMMENDATION: NO-GO**

Stop here until the user explicitly authorizes further work.